# Demonstrating Concurrent Proxy Server Discovery & Probing

This notebook demonstrates using the newly implemented **Concurrent Server Discovery & Probing** feature. 

When a list of proxy server URLs (or a comma-separated string) is supplied to the client, it concurrently probes all candidates in parallel using `concurrent.futures.ThreadPoolExecutor` with a fast 0.5s timeout. It locks onto the first responding active server for the entire process duration, ensuring zero latency overhead and robust fallback handling.

### 1. Initialize Client with Multiple Candidate Server URLs
We initialize a `NorgateDataClient` supplying a list of server URLs where the first one (`port 9999`) is dead/unbound, and the second one (`port 8000`) is our running proxy server:

In [ ]:
import time
from ngd_proxy import NorgateDataClient

candidates = ["http://127.0.0.1:9999", "http://127.0.0.1:8000"]

t0 = time.time()
client = NorgateDataClient(base_url=candidates)
duration_ms = (time.time() - t0) * 1000

print(f"Probing completed in: {duration_ms:.2f} ms")
print(f"Locked Server URL   : {client.base_url}")

### 2. Verify Successful sticky routing
Let's confirm that we can communicate with the selected active server successfully by calling its `/status` endpoint:

In [ ]:
status_data = client.status()
print("Server Status:")
import json
print(json.dumps(status_data, indent=2))

### 3. Verify Fallback Behavior
If none of the supplied server candidates are responding, the client gracefully falls back to the first candidate in the list, emitting a warning in the logs:

In [ ]:
dead_candidates = ["http://127.0.0.1:9998", "http://127.0.0.1:9997"]
client_dead = NorgateDataClient(base_url=dead_candidates)
print(f"Fallback Server URL : {client_dead.base_url}")